In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score
from collections import Counter

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from itertools import combinations

In [30]:
SEED = 26
print(SEED)

26


In [31]:
#verfication du dataset
df = pd.read_csv('data/wdbc.csv')
assert df.shape == (569, 32), "dataset is not complete"
print(df.shape)
df.info()
assert df["sample_id"].duplicated().sum() == 0, "duplicated line in dataset"

assert df.isna().sum().sum() == 0, "null values in dataset"


diagnosis_value = set(df["diagnosis"].unique())
assert diagnosis_value == {'benign', 'malignant'}, "diagnosis column uncomplete"
diagnosis_counts = df["diagnosis"].value_counts()

assert diagnosis_counts["benign"] == 357 and diagnosis_counts["malignant"] == 212, "dataset not accorded to manifest.json"

print("all verifications okay")

(569, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   sample_id                569 non-null    object 
 1   diagnosis                569 non-null    object 
 2   mean_radius              569 non-null    float64
 3   mean_texture             569 non-null    float64
 4   mean_perimeter           569 non-null    float64
 5   mean_area                569 non-null    float64
 6   mean_smoothness          569 non-null    float64
 7   mean_compactness         569 non-null    float64
 8   mean_concavity           569 non-null    float64
 9   mean_concave_points      569 non-null    float64
 10  mean_symmetry            569 non-null    float64
 11  mean_fractal_dimension   569 non-null    float64
 12  radius_error             569 non-null    float64
 13  texture_error            569 non-null    float64
 14  perimeter_error 

In [32]:
#prepration du dataset
label_map = {"benign": 0, "malignant": 1}
y = df["diagnosis"].map(label_map)

assert y.isna().sum() == 0, "y contains NaN"
y_counts = y.value_counts()
assert y_counts[0] == 357, "wrong number of benign"
assert y_counts[1] == 212, "wrong number of malign"

X = df.drop(columns=["sample_id", "diagnosis"])
assert X.shape == (569, 30), "x is not the right size"
assert df.shape == (569, 32), "df has been modified"


In [33]:
#fonctions

def golub_fit(X_train, y_train, k=10): #choix des dix meilleur variables
    mu_benin = X_train[y_train == 0].mean()
    et_benin = X_train[y_train == 0].std()

    mu_malin = X_train[y_train == 1].mean()
    et_malin = X_train[y_train == 1].std()

    w_s = (mu_malin - mu_benin) / (et_malin + et_benin)
    milieu = (mu_malin + mu_benin) / 2

    best_k = w_s.abs().nlargest(k).index

    return best_k, w_s[best_k], milieu[best_k]

def golub_predict(best_k, poids, milieu, X_test): #prediction
    x = X_test[best_k]
    vote = poids * (x - milieu)
    v_x = vote.sum(axis=1)
    y_pred = (v_x >= 0).astype(int)
    return y_pred, v_x

In [34]:
#creation du dataset
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)

resultat = []
selected_sets = []
expl = None

assert rskf.get_n_splits() == 15, "wrong number of splits"
reference = y.mean() #on a que des 0 et 1, on doit avoir 37% de 1
for i, (train_index, test_index) in enumerate(rskf.split(X, y)):
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    assert len(X_train) + len(X_test) == 569, "wrong X split"
    assert len(y_train) + len(y_test) ==  569, "wrong y split"
    assert not (set(train_index) & set(test_index)), "data leak"

    assert abs(y_train.mean() - reference) < 0.01, "wrong y_train stratification"
    assert abs(y_test.mean() - reference) < 0.01, "wrong y_test stratification"

    best_k, poids, milieu = golub_fit(X_train, y_train)
    y_pred, v_x = golub_predict(best_k, poids, milieu, X_test)
    

    selected_sets.append(set(best_k))
    resultat.append({
        "fold": i,
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "auc": roc_auc_score(y_test, v_x)
    })

    if i == 0:
        expl = confusion_matrix(y_test, y_pred) #on garde la matrice que sur le premier pli

df_res = pd.DataFrame(resultat)
print("Matrice de confusion (modèle 0) :\n", expl)
print(df_res[["balanced_accuracy", "auc"]].agg(["mean", "std"]))

compteur = Counter(f for s in selected_sets for f in s)
freq = pd.Series(compteur).reindex(X.columns, fill_value=0).sort_values(ascending=False)
print(freq)

Matrice de confusion (modèle 0) :
 [[70  1]
 [11 32]]
      balanced_accuracy       auc
mean           0.858667  0.960504
std            0.032460  0.017524
mean_radius                15
mean_perimeter             15
mean_area                  15
mean_concavity             15
mean_concave_points        15
worst_radius               15
worst_perimeter            15
worst_area                 15
worst_concave_points       15
worst_concavity            15
mean_compactness            0
mean_smoothness             0
mean_symmetry               0
mean_texture                0
area_error                  0
perimeter_error             0
texture_error               0
radius_error                0
mean_fractal_dimension      0
smoothness_error            0
concavity_error             0
compactness_error           0
worst_texture               0
fractal_dimension_error     0
symmetry_error              0
concave_points_error        0
worst_compactness           0
worst_smoothness            0
wors

In [35]:
compteur = Counter(f for s in selected_sets for f in s)
freq = pd.Series(compteur).reindex(X.columns, fill_value=0).sort_values(ascending=False)
print(freq)

jaccards = [len(a & b) / len(a | b) for a, b in combinations(selected_sets, 2)]
print("Jaccard moyen :", np.mean(jaccards))

mean_radius                15
mean_perimeter             15
mean_area                  15
mean_concavity             15
mean_concave_points        15
worst_radius               15
worst_perimeter            15
worst_area                 15
worst_concave_points       15
worst_concavity            15
mean_compactness            0
mean_smoothness             0
mean_symmetry               0
mean_texture                0
area_error                  0
perimeter_error             0
texture_error               0
radius_error                0
mean_fractal_dimension      0
smoothness_error            0
concavity_error             0
compactness_error           0
worst_texture               0
fractal_dimension_error     0
symmetry_error              0
concave_points_error        0
worst_compactness           0
worst_smoothness            0
worst_symmetry              0
worst_fractal_dimension     0
dtype: int64
Jaccard moyen : 1.0


In [36]:
#reg logistique

rskf_lr = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
resultat_lr = []

for i, (train_index, test_index) in enumerate(rskf_lr.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    clf = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000)
    clf.fit(X_train_s, y_train)

    y_pred = clf.predict(X_test_s)
    proba = clf.predict_proba(X_test_s)[:, 1]

    resultat_lr.append({
        "fold": i,
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "auc": roc_auc_score(y_test, proba)
    })

df_res_lr = pd.DataFrame(resultat_lr)
assert len(df_res_lr) == len(df_res) == 15

comparaison = pd.DataFrame({
    "Golub (10 features)": df_res[["balanced_accuracy", "auc"]].agg(["mean", "std"]).T.stack(),
    "Régression logistique (30 features)": df_res_lr[["balanced_accuracy", "auc"]].agg(["mean", "std"]).T.stack()
})
print(comparaison.round(3))


                        Golub (10 features)  \
balanced_accuracy mean                0.859   
                  std                 0.032   
auc               mean                0.961   
                  std                 0.018   

                        Régression logistique (30 features)  
balanced_accuracy mean                                0.975  
                  std                                 0.016  
auc               mean                                0.994  
                  std                                 0.007  
